In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

Null Check Function

In [0]:

def null_check(df):

    null_counts = []

    for col_name in df.columns:

        cnt = df.filter(
            F.col(col_name).isNull()
        ).count()

        null_counts.append(
            (col_name, cnt)
        )

    return spark.createDataFrame(
        null_counts,
        ["column_name", "null_count"]
    )

Duplicate Check Function

In [0]:
def duplicate_check(df, key_columns):

    duplicate_count = (
        df.groupBy(key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    return duplicate_count

Remove Numeric Characters

In [0]:
def regex_replace_columns(df, columns, pattern, replacement=""):
    for col_name in columns:
        df = df.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), pattern, replacement)
        )
    return df

Null Replacement

In [0]:
def fill_nulls(
    df,
    column_defaults
):

    return df.fillna(
        column_defaults
    )

Stnadlize the Data 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def standardize_string_columns(df):

    # Get all string columns
    string_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]

    # Standardize all string columns
    for col_name in string_cols:

        df = df.withColumn(
            col_name,
            F.initcap(
                F.when(
                    F.trim(F.col(col_name)) == "",
                    None
                ).otherwise(
                    F.regexp_replace(
                        F.trim(F.col(col_name)),
                        "\\s+",
                        " "
                    )
                )
            )
        )

    return df

Primary Key validation

In [0]:


def validate_primary_key(df, primary_keys):

    # Null Check
    null_count = df.filter(
        F.expr(
            " OR ".join(
                [f"{col} IS NULL" for col in primary_keys]
            )
        )
    ).count()

    # Duplicate Check
    duplicate_count = (
        df.groupBy(primary_keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    validation_result = {
        "null_count": null_count,
        "duplicate_count": duplicate_count,
        "status": "PASS"
    }

    if null_count > 0 or duplicate_count > 0:
        validation_result["status"] = "FAIL"

    return validation_result

In [0]:
def run_validations(
    df,
    key_columns
):

    result = {}

    # Duplicate Check
    result["duplicates"] = duplicate_check(
        df,
        key_columns
    )

    # Primary Key Validation
    result["primary_key"] = validate_primary_key(
        df,
        key_columns
    )

    # Null Check
    result["nulls"] = null_check(df)
    #Standlize data
    



    return result